In [1]:
from google.colab import drive
drive.mount('/content/drive/')

# Cambia la directory di lavoro in quella del tuo progetto
# (Assicurati che il percorso sia esattamente quello in cui tieni la cartella eomt sul tuo Drive)
%cd /content/drive/MyDrive/project/semantic-segmentation-roads/eomt

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/semantic-segmentation-roads/eomt


In [3]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 45.9 MB/

In [2]:
import os
import sys
import torch

PROJECT_DIR = "/content/drive/MyDrive/project/semantic-segmentation-roads"
EOMT_DIR = f"{PROJECT_DIR}/eomt"

state_dict_paths = ["/content/drive/MyDrive/project/models_weights/eomt_coco_finetuned.bin", "/content/drive/MyDrive/project/models_weights/3comp_eomt_coco_finetuned.bin", "/content/drive/MyDrive/project/models_weights/head_eomt_coco_finetuned.bin"]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

os.chdir(PROJECT_DIR)

sys.path.insert(0, EOMT_DIR)     # for models.*, datasets.*, training.*
sys.path.insert(0, PROJECT_DIR)  # for eomt.*, utils.*
from eomt.semantic_eval import evaluate_semantic
from utils.model_loading import get_config, build_model, load_weights
from utils.data_loading import build_datamodule

In [3]:
config_cs = get_config()
data = build_datamodule(config_cs)
eval_dataset_cs, img_size_cs, num_classes_cs = data.val_dataloader(), data.img_size, data.num_classes


In [5]:
for path in state_dict_paths:
  print("="*30)
  print(f"Evaluating {path}")
  model_cs = build_model(config_cs, (640, 640), num_classes_cs, masked_attn_enabled=True).eval().to(device)
  model_cs = load_weights(model_cs, path, device)
  evaluate_semantic(model_cs, eval_dataset_cs, device)

Evaluating /content/drive/MyDrive/project/models_weights/eomt_coco_finetuned.bin


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Missing keys: []
Unexpected keys: []
Evaluating on 500 images...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]


mIoU: 72.05%
Evaluating /content/drive/MyDrive/project/models_weights/3comp_eomt_coco_finetuned.bin
Missing keys: []
Unexpected keys: []
Evaluating on 500 images...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]


mIoU: 71.68%
Evaluating /content/drive/MyDrive/project/models_weights/head_eomt_coco_finetuned.bin
Missing keys: []
Unexpected keys: []
Evaluating on 500 images...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]


mIoU: 57.25%
